<a href="https://colab.research.google.com/github/melitta1114/melitta1114.github.io/blob/main/ABDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [66]:
import pandas as pd
import numpy as np

In [67]:
df=pd.read_excel("FSEGA_Baza_De_Date_ADBF.xlsx", engine='openpyxl')

In [68]:
df

,ItemID,SalesDate,CostPrice,QtySales
0,SKU_1331_111,2017-01-01,56.12,73.0
1,SKU_1331_111,2017-02-01,56.12,11.0
2,SKU_1331_111,2017-03-01,56.12,72.0
3,SKU_1331_111,2017-04-01,56.12,96.0
4,SKU_1331_111,2017-05-01,56.12,27.0
...,...,...,...,...
84588,SKU_3781_237,2024-06-01,314.22,8.0
84589,SKU_3781_237,2024-07-01,314.22,23.0
84590,SKU_3781_237,2024-08-01,314.22,21.0
84591,SKU_3781_237,2024-09-01,314.22,39.0


In [69]:
print(df.info())
print(df.dtypes)
print(df.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84593 entries, 0 to 84592
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   ItemID     84593 non-null  object        
 1   SalesDate  84593 non-null  datetime64[ns]
 2   CostPrice  84593 non-null  float64       
 3   QtySales   84593 non-null  float64       
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 2.6+ MB
None
ItemID               object
SalesDate    datetime64[ns]
CostPrice           float64
QtySales            float64
dtype: object
ItemID       1003
SalesDate      94
CostPrice     938
QtySales     7992
dtype: int64


In [70]:
duplications = df[df.duplicated()]

In [71]:
duplications

,ItemID,SalesDate,CostPrice,QtySales


In [72]:
df["Revenue"] = df["QtySales"] * df["CostPrice"]

In [73]:
df["ItemGroup"] = df["ItemID"].str[-3:]

In [74]:
df

,ItemID,SalesDate,CostPrice,QtySales,Revenue,ItemGroup
0,SKU_1331_111,2017-01-01,56.12,73.0,4096.76,111
1,SKU_1331_111,2017-02-01,56.12,11.0,617.32,111
2,SKU_1331_111,2017-03-01,56.12,72.0,4040.64,111
3,SKU_1331_111,2017-04-01,56.12,96.0,5387.52,111
4,SKU_1331_111,2017-05-01,56.12,27.0,1515.24,111
...,...,...,...,...,...,...
84588,SKU_3781_237,2024-06-01,314.22,8.0,2513.76,237
84589,SKU_3781_237,2024-07-01,314.22,23.0,7227.06,237
84590,SKU_3781_237,2024-08-01,314.22,21.0,6598.62,237
84591,SKU_3781_237,2024-09-01,314.22,39.0,12254.58,237


In [75]:
def calculate_seasonality(df):
    df = df.copy()

    df['YearMonth'] = df['SalesDate'].dt.to_period('M')
    df['Month'] = df['SalesDate'].dt.month

    monthly_sales = (
        df.groupby(['ItemID', 'YearMonth', 'Month'])
        .agg({'Revenue': 'sum'})
        .reset_index()
    )

    avg_monthly_sales = (
        monthly_sales.groupby('ItemID')['Revenue']
        .mean()
        .reset_index()
        .rename(columns={'Revenue': 'AvgMonthlyRevenue'})
    )
    avg_monthly_sales.rename(columns={'Revenue':'AvgMonthlyRevenue'}, inplace=True)

    monthly_sales = monthly_sales.merge(avg_monthly_sales, on='ItemID', how='left')

    monthly_sales['SeasonalDeviation'] = monthly_sales['Revenue'] - monthly_sales['AvgMonthlyRevenue']


    top_months = (
        monthly_sales.sort_values(['ItemID', 'SeasonalDeviation'], ascending=[True, False])
        .groupby('ItemID')
        .head(6)
    )

    def longest_consecutive_range(month_list):
        months = sorted(set(month_list))
        longest = []
        current = [months[0]]

        for i in range(1, len(months)):
            if months[i] == months[i - 1] + 1:
                current.append(months[i])
            else:
                if len(current) > len(longest):
                    longest = current
                current = [months[i]]
        if len(current) > len(longest):
            longest = current
        if len(longest) < 2:
            return ''
        return f"{longest[0]}-{longest[-1]}"

    seasonality_df = (
        top_months.groupby('ItemID')['Month']
        .apply(lambda months: longest_consecutive_range(months))
        .reset_index()
        .rename(columns={'Month': 'SeasonalityHint'})
    )


    return seasonality_df

In [76]:
df2 = calculate_seasonality(df)

In [77]:
df2

,ItemID,SeasonalityHint
0,SKU_1009_212,8-10
1,SKU_1012_149,1-2
2,SKU_1015_128,3-5
3,SKU_1017_370,2-3
4,SKU_1019_426,7-8
...,...,...
998,SKU_9909_291,8-10
999,SKU_9920_400,10-12
1000,SKU_9938_134,9-10
1001,SKU_9971_489,10-12


In [78]:
revenues = df.groupby("ItemID").agg(
    TotalRevenue = ("Revenue", "sum"),
    Volatility = ("Revenue", "std"),
    FirstSale = ("SalesDate", "min"),
    LastSale = ("SalesDate", "max"),
).reset_index()
revenues["ItemGroup"] = revenues["ItemID"].str[-3:]
revenues["SeasonalityHint"] = df2["SeasonalityHint"]

In [79]:
revenues

,ItemID,TotalRevenue,Volatility,FirstSale,LastSale,ItemGroup,SeasonalityHint
0,SKU_1009_212,2554599.02,28188.893436,2017-01-01,2024-10-01,212,8-10
1,SKU_1012_149,4120.69,91.847739,2021-07-01,2024-10-01,149,1-2
2,SKU_1015_128,150499.44,1670.629994,2017-03-01,2024-10-01,128,3-5
3,SKU_1017_370,28188.30,305.273633,2017-01-01,2024-10-01,370,2-3
4,SKU_1019_426,57856.00,458.186508,2017-01-01,2024-10-01,426,7-8
...,...,...,...,...,...,...,...
998,SKU_9909_291,65105.16,773.222113,2017-01-01,2024-10-01,291,8-10
999,SKU_9920_400,196661.50,3248.785894,2017-01-01,2024-10-01,400,10-12
1000,SKU_9938_134,58564.66,596.260324,2018-10-01,2024-10-01,134,9-10
1001,SKU_9971_489,40295.45,202.753979,2017-09-01,2024-10-01,489,10-12


In [80]:
groups = df["ItemGroup"].nunique()
groups

385

In [81]:
revenues_sorted = revenues.sort_values(by=["TotalRevenue"], ascending=[False])

In [82]:
revenues_sorted

,ItemID,TotalRevenue,Volatility,FirstSale,LastSale,ItemGroup,SeasonalityHint
951,SKU_9481_168,1.636551e+07,64775.617707,2017-01-01,2024-10-01,168,9-10
419,SKU_4623_274,1.218741e+07,128357.018555,2017-01-01,2024-10-01,274,3-5
118,SKU_1899_112,1.206623e+07,56788.811531,2017-01-01,2024-10-01,112,3-4
246,SKU_3073_202,1.103111e+07,31092.793549,2017-01-01,2024-10-01,202,5-9
718,SKU_7360_219,1.082120e+07,116093.830112,2017-07-01,2024-10-01,219,8-11
...,...,...,...,...,...,...,...
755,SKU_7784_473,6.019200e+02,20.583590,2022-06-01,2024-10-01,473,3-5
146,SKU_2148_177,5.012800e+02,7.086838,2020-10-01,2024-10-01,177,
280,SKU_3423_427,4.967750e+02,7.278006,2020-05-01,2024-10-01,427,9-12
682,SKU_7019_288,1.954400e+02,2.389498,2017-02-01,2024-10-01,288,3-6


In [83]:
def groups(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df.sort_values("TotalRevenue", ascending=False).reset_index(drop=True)

    df["CumulativeRevenue"] = df["TotalRevenue"].cumsum()
    total_revenue = df["TotalRevenue"].sum()
    df["CumulativeShare"] = df["CumulativeRevenue"] / total_revenue

    def assign_abc(cum_share):
        if cum_share <= 0.80:
            return "A"
        elif cum_share <= 0.95:
            return "B"
        else:
            return "C"

    df["ABC_Category"] = df["CumulativeShare"].apply(assign_abc)

    df["IsNew"] = df["FirstSale"] > pd.to_datetime("2023-10-01")

    df["IsObsolete"] = df["LastSale"] < pd.to_datetime("2024-01-01")

    df["VolatilityLevel"] = pd.qcut(df["Volatility"], q=3, labels=["Low", "Medium", "High"])

    return df

In [84]:
result = groups(revenues_sorted)

In [85]:
result

,ItemID,TotalRevenue,Volatility,FirstSale,LastSale,ItemGroup,SeasonalityHint,CumulativeRevenue,CumulativeShare,ABC_Category,IsNew,IsObsolete,VolatilityLevel
0,SKU_9481_168,1.636551e+07,64775.617707,2017-01-01,2024-10-01,168,9-10,1.636551e+07,0.031390,A,False,False,High
1,SKU_4623_274,1.218741e+07,128357.018555,2017-01-01,2024-10-01,274,3-5,2.855291e+07,0.054767,A,False,False,High
2,SKU_1899_112,1.206623e+07,56788.811531,2017-01-01,2024-10-01,112,3-4,4.061914e+07,0.077911,A,False,False,High
3,SKU_3073_202,1.103111e+07,31092.793549,2017-01-01,2024-10-01,202,5-9,5.165025e+07,0.099070,A,False,False,High
4,SKU_7360_219,1.082120e+07,116093.830112,2017-07-01,2024-10-01,219,8-11,6.247145e+07,0.119826,A,False,False,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...
998,SKU_7784_473,6.019200e+02,20.583590,2022-06-01,2024-10-01,473,3-5,5.213508e+08,0.999997,C,False,False,Low
999,SKU_2148_177,5.012800e+02,7.086838,2020-10-01,2024-10-01,177,,5.213514e+08,0.999998,C,False,False,Low
1000,SKU_3423_427,4.967750e+02,7.278006,2020-05-01,2024-10-01,427,9-12,5.213518e+08,0.999999,C,False,False,Low
1001,SKU_7019_288,1.954400e+02,2.389498,2017-02-01,2024-10-01,288,3-6,5.213520e+08,1.000000,C,False,False,Low


In [86]:
print(result["ABC_Category"].value_counts())

ABC_Category
C    550
B    268
A    185
Name: count, dtype: int64


In [87]:
result.to_csv('result.csv', index=False, encoding='utf-8')

from google.colab import files
files.download('result.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [88]:
import plotly.express as px

fig = px.pie(result, names="ABC_Category", title="ABC kategóriák eloszlása")
fig.show()

In [89]:
fig = px.scatter(
    result,
    x="Volatility",
    y="TotalRevenue",
    size="CumulativeShare",
    color="ABC_Category",
    hover_name="ItemID",
    title="Volatilitás és Árbevétel összefüggése",
    width=1200,   # szélesség pixelben
    height=800
)
fig.show()

In [90]:
def explode_seasonality(df):
    records = []
    for _, row in df.iterrows():
        item_group = row['ItemGroup']
        abc_cat = row['ABC_Category']
        seasonality = row['SeasonalityHint']
        if pd.isna(seasonality) or seasonality.strip() == "":
            continue
        try:
            parts = seasonality.split('-')
            start, end = int(parts[0]), int(parts[1])
            if start <= end:
                months = list(range(start, end + 1))
            else:
                months = list(range(start, 13)) + list(range(1, end + 1))
        except:
            continue
        for month in months:
            records.append({
                'ItemGroup': item_group,
                'Month': month,
                'ABC_Category': abc_cat
            })
    return pd.DataFrame(records)

In [91]:
import plotly.graph_objects as go

season_df = explode_seasonality(result)

heatmap_data = (
    season_df.groupby(['ABC_Category', 'ItemGroup', 'Month'])
    .size()
    .reset_index(name='SeasonalProductCount')
)

categories = heatmap_data['ABC_Category'].unique()

fig_data = []
buttons = []

for i, cat in enumerate(categories):
    df_cat = heatmap_data[heatmap_data['ABC_Category'] == cat]
    pivot = df_cat.pivot(index='ItemGroup', columns='Month', values='SeasonalProductCount').fillna(0)
    z = pivot.values
    x = ['Jan', 'Feb', 'Már', 'Ápr', 'Máj', 'Jún', 'Júl', 'Aug', 'Szep', 'Okt', 'Nov', 'Dec']
    y = pivot.index.astype(str).tolist()

    heatmap = go.Heatmap(
        z=z,
        x=x,
        y=y,
        colorscale='Viridis',
        visible=(i == 0),
        colorbar=dict(title="Szezonális termékek száma")
    )
    fig_data.append(heatmap)

    buttons.append(dict(
        label=f"Kategória {cat}",
        method="update",
        args=[{"visible": [j == i for j in range(len(categories))]},
              {"title": f"Szezonális hónapok hőtérképe - {cat} kategória"}]
    ))

updatemenus = [
    dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0.5,
        xanchor="center",
        y=1.1,
        yanchor="top"
    )
]

fig = go.Figure(data=fig_data)
fig.update_layout(
    title="Szezonális hónapok hőtérképe - A kategória",
    updatemenus=updatemenus,
    width=1200,
    height=1000,
    xaxis_title="Hónap",
    yaxis_title="Termékcsoport"
)

fig.show()


In [92]:
result_sorted = result.sort_values("CumulativeRevenue")
fig = px.bar(
    result_sorted,
    x="ItemID",
    y="CumulativeRevenue",
    color="ABC_Category",
    title="Cumulative Revenue per Product"
)
fig.show()

In [93]:
result['Turnover'] = result['TotalRevenue'] / ((result['LastSale'] - result['FirstSale']).dt.days / 30)

fig = px.box(result,
             x="ABC_Category",
             y="Turnover",
             color="VolatilityLevel",
             title="Kategóriánkénti becsült forgási sebesség",
             height= 1000
)
fig.show()

In [94]:
def season_length(s):
    if pd.isna(s) or s.strip() == "":
        return 0
    try:
        start, end = map(int, s.split("-"))
        return abs(end - start)
    except:
        return 0

result['SeasonLength'] = result['SeasonalityHint'].apply(season_length)
result['StockoutRisk'] = result['Volatility'] * result['SeasonLength']

fig = px.histogram(
    result,
    x="StockoutRisk",
    color="ABC_Category",
    nbins=50,
    title="Potenciális készlethiányos kockázatok eloszlása",
    labels={"StockoutRisk": "Volatilitás × Szezonhossz"}
)
fig.show()

In [95]:
today = pd.to_datetime("2024-10-01")
result['DeadStock'] = (today - result['LastSale']).dt.days > 365
deadstock_share = result.groupby('ABC_Category')['DeadStock'].mean().reset_index()

fig = px.bar(deadstock_share,
             x="ABC_Category",
             y="DeadStock",
             title="Túlzott készlet aránya (Deadstock)",
             labels={"DeadStock": "Arány"})
fig.show()